# 🔺 Taller - Cálculo y Visualización de Normales

**Objetivo:** Calcular vectores normales de superficies 3D y utilizarlos para iluminación correcta. Comprender la diferencia entre normales de vértices y caras, smooth shading vs flat shading, y visualizar normales para debugging.

**Herramientas:** `trimesh`, `numpy`, `matplotlib`, `vedo`

---
**Contenido:**
1. Instalación y configuración
2. Carga del modelo 3D
3. Cálculo de normales de caras
4. Cálculo de normales de vértices
5. Flat Shading vs Smooth Shading
6. Validación de normales
7. Visualización 3D interactiva con Vedo

## 📦 Sección 1 — Instalación y configuración

In [1]:
!pip install trimesh numpy matplotlib vedo pygltflib --quiet

print('✅ Dependencias instaladas.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 24.8 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.5/740.5 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
✅ Dependencias instaladas.


In [2]:
# ══════════════════════════════════════════════════════════
# CONFIGURACIÓN — ajusta esta variable con el nombre
# sonic_model.glb subido a Colab
# ══════════════════════════════════════════════════════════
MODELO_BASE = 'sonic_model.glb'

print(f'📁 Modelo configurado: {MODELO_BASE}')

📁 Modelo configurado: sonic_model.glb


In [3]:
import numpy as np
import trimesh
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
import warnings
import gc
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor' : '#0f1117',
    'axes.facecolor'   : '#1a1d27',
    'axes.edgecolor'   : '#3a3f55',
    'axes.labelcolor'  : '#c8cce0',
    'xtick.color'      : '#6a6f8a',
    'ytick.color'      : '#6a6f8a',
    'text.color'       : '#c8cce0',
    'grid.color'       : '#2a2f45',
    'grid.alpha'       : 0.5,
    'font.family'      : 'monospace',
})

print('✅ Importaciones listas.')

✅ Importaciones listas.


## 📂 Sección 2 — Carga del modelo 3D

Sube tu archivo `.glb` o `.gltf` usando el panel de archivos de Colab (ícono de carpeta a la izquierda).

In [4]:
# Cargamos el modelo — trimesh maneja GLTF/GLB automáticamente
escena_o_malla = trimesh.load(MODELO_BASE)

# Los archivos GLTF pueden contener múltiples mallas (escena)
# Los combinamos en una sola malla para análisis uniforme
if isinstance(escena_o_malla, trimesh.Scene):
    geometrias = list(escena_o_malla.geometry.values())
    if len(geometrias) == 1:
        mesh = geometrias[0]
    else:
        # Concatenamos todas las geometrías de la escena
        mesh = trimesh.util.concatenate(geometrias)
    print(f'📦 Escena GLTF con {len(geometrias)} geometría(s) — combinadas en una malla.')
else:
    mesh = escena_o_malla
    print('📦 Malla única cargada directamente.')

print()
print(f'   Vértices : {len(mesh.vertices):,}')
print(f'   Caras    : {len(mesh.faces):,}')
print(f'   Watertight: {mesh.is_watertight}')
print(f'   Bounds   : min {mesh.bounds[0].round(3)}  max {mesh.bounds[1].round(3)}')

ValueError: string is not a file: `sonic_model.glb`

## 🔺 Sección 3 — Cálculo de Normales de Caras

La normal de una cara triangular se obtiene con el **producto cruz** de dos de sus aristas.
El resultado es un vector perpendicular al plano del triángulo.

$$\vec{n} = \frac{(B - A) \times (C - A)}{|(B - A) \times (C - A)|}$$

La dirección (hacia afuera o adentro) depende del orden de los vértices (winding order).

In [ ]:
# ── Cálculo manual de normales de caras ──────────────────────
vertices = mesh.vertices   # (N, 3)
caras    = mesh.faces      # (M, 3) — índices de vértices

# Extraemos los tres vértices de cada triángulo
A = vertices[caras[:, 0]]  # (M, 3)
B = vertices[caras[:, 1]]  # (M, 3)
C = vertices[caras[:, 2]]  # (M, 3)

# Vectores de las dos aristas del triángulo
v1 = B - A   # arista AB
v2 = C - A   # arista AC

# Producto cruz: perpendicular al plano del triángulo
normales_caras_raw = np.cross(v1, v2)   # (M, 3)

# Normalizamos: magnitud = 1
magnitudes = np.linalg.norm(normales_caras_raw, axis=1, keepdims=True)
# Evitamos división por cero en caras degeneradas
magnitudes = np.where(magnitudes == 0, 1, magnitudes)
normales_caras_manual = normales_caras_raw / magnitudes

# ── Comparación con trimesh ───────────────────────────────────
normales_caras_trimesh = mesh.face_normals   # calculadas por trimesh

# Diferencia máxima entre nuestro cálculo y el de trimesh
diff = np.max(np.abs(normales_caras_manual - normales_caras_trimesh))

print('🔺 Normales de Caras — Cálculo Manual')
print('=' * 45)
print(f'   Total de caras             : {len(caras):,}')
print(f'   Shape normales manuales    : {normales_caras_manual.shape}')
print(f'   Shape normales trimesh     : {normales_caras_trimesh.shape}')
print(f'   Diferencia máxima (manual vs trimesh): {diff:.2e}')
print()
print('   Primeras 5 normales de caras (manual):')
for i in range(min(5, len(normales_caras_manual))):
    n = normales_caras_manual[i]
    print(f'   Cara {i}: ({n[0]:+.4f}, {n[1]:+.4f}, {n[2]:+.4f})  |mag|={np.linalg.norm(n):.6f}')

In [ ]:
# ── Visualización: normales de caras como flechas ────────────
# Usamos una muestra de caras para no saturar la gráfica
N_MUESTRA = min(300, len(caras))
idx_muestra = np.random.choice(len(caras), N_MUESTRA, replace=False)

# Centro de cada cara seleccionada
centros = (A[idx_muestra] + B[idx_muestra] + C[idx_muestra]) / 3.0
normales_muestra = normales_caras_manual[idx_muestra]

# Longitud de las flechas proporcional al tamaño del modelo
escala = mesh.scale * 0.03

fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(111, projection='3d')
ax.set_facecolor('#1a1d27')

# Flechas de normales
ax.quiver(
    centros[:, 0], centros[:, 1], centros[:, 2],
    normales_muestra[:, 0] * escala,
    normales_muestra[:, 1] * escala,
    normales_muestra[:, 2] * escala,
    color='#ff7043', alpha=0.7, linewidth=0.6
)

# Contorno del modelo (aristas de muestra)
for cara in caras[idx_muestra[:50]]:
    tri = vertices[cara]
    tri = np.vstack([tri, tri[0]])  # cerrar el triángulo
    ax.plot(tri[:,0], tri[:,1], tri[:,2], color='#4fc3f7', alpha=0.2, linewidth=0.4)

ax.set_title(f'🔺 Normales de Caras ({N_MUESTRA} muestras)', fontsize=13, pad=12)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.tight_layout()
plt.savefig('normales_caras.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Visualización de normales de caras generada.')

## 🔵 Sección 4 — Cálculo de Normales de Vértices

La normal de un **vértice** se obtiene promediando las normales de todas las caras que lo comparten:

$$\vec{n}_v = \frac{\sum_{f \in F(v)} \vec{n}_f}{|\sum_{f \in F(v)} \vec{n}_f|}$$

Esto produce transiciones suaves entre caras, base del **smooth shading**.

In [ ]:
# ── Cálculo manual de normales de vértices ───────────────────
n_vertices = len(vertices)
normales_vertices_manual = np.zeros((n_vertices, 3), dtype=np.float64)

# Para cada cara, acumulamos su normal en cada uno de sus 3 vértices
for i, cara in enumerate(caras):
    for idx_v in cara:
        normales_vertices_manual[idx_v] += normales_caras_manual[i]

# Normalizamos cada normal de vértice
magnitudes_v = np.linalg.norm(normales_vertices_manual, axis=1, keepdims=True)
magnitudes_v = np.where(magnitudes_v == 0, 1, magnitudes_v)
normales_vertices_manual /= magnitudes_v

# ── Comparación con trimesh ───────────────────────────────────
normales_vertices_trimesh = mesh.vertex_normals
diff_v = np.max(np.abs(normales_vertices_manual - normales_vertices_trimesh))

print('🔵 Normales de Vértices — Cálculo Manual')
print('=' * 45)
print(f'   Total de vértices          : {n_vertices:,}')
print(f'   Shape normales manuales    : {normales_vertices_manual.shape}')
print(f'   Shape normales trimesh     : {normales_vertices_trimesh.shape}')
print(f'   Diferencia máxima (manual vs trimesh): {diff_v:.2e}')
print()
print('   Primeros 5 vértices:')
for i in range(min(5, n_vertices)):
    nm = normales_vertices_manual[i]
    nt = normales_vertices_trimesh[i]
    print(f'   V{i} manual : ({nm[0]:+.4f}, {nm[1]:+.4f}, {nm[2]:+.4f})')
    print(f'   V{i} trimesh: ({nt[0]:+.4f}, {nt[1]:+.4f}, {nt[2]:+.4f})')
    print()

In [ ]:
# ── Visualización: normales de vértices como flechas ─────────
N_MUESTRA_V = min(300, n_vertices)
idx_v_muestra = np.random.choice(n_vertices, N_MUESTRA_V, replace=False)

verts_muestra  = vertices[idx_v_muestra]
normales_v_m   = normales_vertices_manual[idx_v_muestra]

fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(111, projection='3d')
ax.set_facecolor('#1a1d27')

ax.quiver(
    verts_muestra[:, 0], verts_muestra[:, 1], verts_muestra[:, 2],
    normales_v_m[:, 0] * escala,
    normales_v_m[:, 1] * escala,
    normales_v_m[:, 2] * escala,
    color='#81c784', alpha=0.7, linewidth=0.6
)

ax.scatter(verts_muestra[:,0], verts_muestra[:,1], verts_muestra[:,2],
           color='#ffcc02', s=4, alpha=0.5)

ax.set_title(f'🔵 Normales de Vértices ({N_MUESTRA_V} muestras)', fontsize=13, pad=12)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.tight_layout()
plt.savefig('normales_vertices.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Visualización de normales de vértices generada.')

## 💡 Sección 5 — Flat Shading vs Smooth Shading

| | Flat Shading | Smooth Shading |
|---|---|---|
| Normal usada | Normal de cara (una por triángulo) | Normal de vértice (interpolada) |
| Aspecto | Facetado, se ven los triángulos | Suave, transiciones continuas |
| Uso | Objetos con aristas duras, low-poly | Objetos orgánicos, superficies curvas |

Lo simulamos coloreando cada cara/vértice según la componente Z de su normal,
que representa cuánto apunta hacia la cámara (similar a iluminación difusa).

In [ ]:
# ── Selección de muestra de caras para visualizar ────────────
N_VIZ = min(1500, len(caras))
idx_viz = np.random.choice(len(caras), N_VIZ, replace=False)

caras_viz    = caras[idx_viz]
A_viz = vertices[caras_viz[:, 0]]
B_viz = vertices[caras_viz[:, 1]]
C_viz = vertices[caras_viz[:, 2]]
centros_viz  = (A_viz + B_viz + C_viz) / 3.0

# ── Color por normal Z ────────────────────────────────────────
# Flat: una normal por cara → color uniforme por triángulo
nz_flat   = normales_caras_manual[idx_viz, 2]           # componente Z de la cara
color_flat = cm.coolwarm((nz_flat + 1) / 2)             # mapeamos [-1,1] → [0,1]

# Smooth: normal interpolada en el centroide desde los 3 vértices
nv_A = normales_vertices_manual[caras_viz[:, 0]]
nv_B = normales_vertices_manual[caras_viz[:, 1]]
nv_C = normales_vertices_manual[caras_viz[:, 2]]
normal_interpolada = (nv_A + nv_B + nv_C) / 3.0
nz_smooth   = normal_interpolada[:, 2]
color_smooth = cm.coolwarm((nz_smooth + 1) / 2)

# ── Visualización lado a lado ─────────────────────────────────
fig = plt.figure(figsize=(14, 6))

# Panel izquierdo: Flat Shading
ax1 = fig.add_subplot(121, projection='3d')
ax1.set_facecolor('#1a1d27')
ax1.scatter(centros_viz[:,0], centros_viz[:,1], centros_viz[:,2],
            c=color_flat, s=2, alpha=0.8)
ax1.set_title('🔲 Flat Shading\n(normal de cara — facetado)', fontsize=11, pad=10)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# Panel derecho: Smooth Shading
ax2 = fig.add_subplot(122, projection='3d')
ax2.set_facecolor('#1a1d27')
ax2.scatter(centros_viz[:,0], centros_viz[:,1], centros_viz[:,2],
            c=color_smooth, s=2, alpha=0.8)
ax2.set_title('🟢 Smooth Shading\n(normal de vértice — suave)', fontsize=11, pad=10)
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

fig.suptitle('💡 Flat Shading vs Smooth Shading', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('flat_vs_smooth.png', dpi=120, bbox_inches='tight')
plt.show()

print('📝 Diferencia clave:')
print('   Flat   → cada triángulo tiene un color uniforme (se ven los bordes)')
print('   Smooth → el color varía dentro de cada triángulo (transición continua)')

## ✅ Sección 6 — Validación de Normales

In [ ]:
import pandas as pd

print('✅ Validación de Normales de Caras')
print('=' * 50)

# ── 1. Verificar magnitud = 1 ─────────────────────────────────
magnitudes_final = np.linalg.norm(normales_caras_manual, axis=1)
TOL = 1e-5
no_unitarias = np.sum(np.abs(magnitudes_final - 1.0) > TOL)
print(f'   Normales con |n| ≠ 1 (tolerancia {TOL}): {no_unitarias}')
print(f'   Magnitud mínima : {magnitudes_final.min():.6f}')
print(f'   Magnitud máxima : {magnitudes_final.max():.6f}')
print(f'   Magnitud media  : {magnitudes_final.mean():.6f}')
print()

# ── 2. Detectar normales invertidas ──────────────────────────
# Una normal apunta hacia afuera si su producto punto con el
# vector (centroide_cara - centroide_modelo) es positivo
centroide_modelo = mesh.centroid
centros_caras    = (vertices[caras[:,0]] + vertices[caras[:,1]] + vertices[caras[:,2]]) / 3.0
vectores_afuera  = centros_caras - centroide_modelo

# Producto punto: positivo = apunta afuera, negativo = invertida
dot_products = np.einsum('ij,ij->i', normales_caras_manual, vectores_afuera)
n_invertidas = np.sum(dot_products < 0)
pct_invertidas = n_invertidas / len(caras) * 100

print(f'   Normales invertidas detectadas: {n_invertidas} ({pct_invertidas:.1f}%)')
print()

# ── 3. Corrección de normales invertidas ─────────────────────
normales_corregidas = normales_caras_manual.copy()
normales_corregidas[dot_products < 0] *= -1
print(f'   Normales corregidas: {n_invertidas} invertidas → orientación corregida')
print()

# ── 4. Consistencia de orientación ───────────────────────────
# Verificamos que normales de caras adyacentes sean similares
# (diferencia de ángulo pequeña = orientación consistente)
# Tomamos muestra de pares de caras adyacentes
if hasattr(mesh, 'face_adjacency') and len(mesh.face_adjacency) > 0:
    adj = mesh.face_adjacency[:500]  # primeras 500 adyacencias
    n1  = normales_caras_manual[adj[:, 0]]
    n2  = normales_caras_manual[adj[:, 1]]
    dot_adj = np.einsum('ij,ij->i', n1, n2)
    angulos = np.degrees(np.arccos(np.clip(dot_adj, -1, 1)))
    print(f'   Consistencia (caras adyacentes):')
    print(f'     Ángulo medio entre normales adyacentes : {angulos.mean():.1f}°')
    print(f'     Ángulo máximo entre normales adyacentes: {angulos.max():.1f}°')
    print(f'     Caras con ángulo > 90° (posible error) : {np.sum(angulos > 90)}')

print()

# ── Tabla resumen ─────────────────────────────────────────────
resumen = pd.DataFrame({
    'Métrica': [
        'Total caras',
        'Normales unitarias (|n|=1)',
        'Normales NO unitarias',
        'Normales invertidas',
        'Normales correctas',
    ],
    'Valor': [
        len(caras),
        len(caras) - no_unitarias,
        no_unitarias,
        n_invertidas,
        len(caras) - n_invertidas,
    ]
})
print(resumen.to_string(index=False))

In [ ]:
# ── Visualización: normales correctas vs invertidas ───────────
N_VAL = min(400, len(caras))
idx_val = np.random.choice(len(caras), N_VAL, replace=False)

centros_val  = centros_caras[idx_val]
normales_val = normales_caras_manual[idx_val]
dot_val      = dot_products[idx_val]

# Colores: verde = correcta, rojo = invertida
colores_val = np.where(dot_val >= 0, '#81c784', '#ff5252')

fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(111, projection='3d')
ax.set_facecolor('#1a1d27')

# Flechas coloreadas según orientación
for i in range(len(centros_val)):
    ax.quiver(
        centros_val[i,0], centros_val[i,1], centros_val[i,2],
        normales_val[i,0]*escala, normales_val[i,1]*escala, normales_val[i,2]*escala,
        color=colores_val[i], alpha=0.7, linewidth=0.5
    )

# Leyenda manual
from matplotlib.lines import Line2D
leyenda = [
    Line2D([0],[0], color='#81c784', linewidth=2, label='Normal correcta (apunta afuera)'),
    Line2D([0],[0], color='#ff5252', linewidth=2, label='Normal invertida'),
]
ax.legend(handles=leyenda, loc='upper left', fontsize=9,
          facecolor='#1a1d27', edgecolor='#3a3f55', labelcolor='#c8cce0')

ax.set_title('✅ Validación — Normales correctas (verde) vs invertidas (rojo)',
             fontsize=11, pad=12)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.tight_layout()
plt.savefig('validacion_normales.png', dpi=120, bbox_inches='tight')
plt.show()

## 🌐 Sección 7 — Visualización 3D Interactiva con Vedo

Vedo permite renderizar el modelo con iluminación real usando las normales calculadas,
comparar flat vs smooth shading en 3D, y visualizar las flechas de normales en perspectiva.

In [ ]:
# Configuramos vedo para funcionar en Colab (modo no interactivo)
import vedo
vedo.settings.default_backend = 'vtk'

print(f'✅ Vedo {vedo.__version__} listo.')

In [ ]:
# ── Flat Shading con Vedo ─────────────────────────────────────
# Convertimos la malla trimesh a vedo
malla_vedo = vedo.Mesh([mesh.vertices, mesh.faces])

# Flat shading: cada cara tiene su propia normal
malla_flat = malla_vedo.clone()
malla_flat.flat()                          # activa flat shading
malla_flat.cmap('coolwarm', normales_caras_manual[:, 2], on='cells')

# Smooth shading: normales interpoladas entre vértices
malla_smooth = malla_vedo.clone()
malla_smooth.phong()                       # activa smooth shading
malla_smooth.cmap('coolwarm', normales_vertices_manual[:, 2], on='points')

# Renderizamos y guardamos como imágenes
plt_flat = vedo.Plotter(offscreen=True, size=(600, 500))
plt_flat.add(malla_flat)
plt_flat.show()
plt_flat.screenshot('vedo_flat.png')
plt_flat.close()

plt_smooth = vedo.Plotter(offscreen=True, size=(600, 500))
plt_smooth.add(malla_smooth)
plt_smooth.show()
plt_smooth.screenshot('vedo_smooth.png')
plt_smooth.close()

# Mostramos las imágenes en matplotlib
img_flat   = plt.imread('vedo_flat.png')
img_smooth = plt.imread('vedo_smooth.png')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(img_flat)
axes[0].set_title('🔲 Flat Shading (Vedo)', fontsize=12)
axes[0].axis('off')
axes[1].imshow(img_smooth)
axes[1].set_title('🟢 Smooth Shading (Vedo)', fontsize=12)
axes[1].axis('off')
fig.suptitle('💡 Flat vs Smooth Shading — Vedo', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('vedo_flat_vs_smooth.png', dpi=120, bbox_inches='tight')
plt.show()

gc.collect()
print('✅ Comparación Flat vs Smooth con Vedo generada.')

In [ ]:
# ── Visualización de normales como flechas en Vedo ───────────
N_FLECHAS = min(200, len(caras))
idx_f = np.random.choice(len(caras), N_FLECHAS, replace=False)

centros_f  = centros_caras[idx_f]
normales_f = normales_caras_manual[idx_f]
escala_f   = mesh.scale * 0.04

# Creamos las flechas en vedo
flechas = vedo.Arrows(
    centros_f,
    centros_f + normales_f * escala_f,
    c='#ff7043',
    alpha=0.8
)

malla_base = vedo.Mesh([mesh.vertices, mesh.faces])
malla_base.color('#4fc3f7').alpha(0.3)

plt_flechas = vedo.Plotter(offscreen=True, size=(700, 550))
plt_flechas.add(malla_base, flechas)
plt_flechas.show()
plt_flechas.screenshot('vedo_normales_flechas.png')
plt_flechas.close()

img_flechas = plt.imread('vedo_normales_flechas.png')
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(img_flechas)
ax.set_title('🔺 Normales de caras como flechas — Vedo', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig('vedo_normales_flechas_final.png', dpi=120, bbox_inches='tight')
plt.show()

gc.collect()
print('✅ Visualización de flechas con Vedo generada.')

In [ ]:
# ── Empaquetamos todos los outputs para descargar ─────────────
import zipfile, os

archivos = [
    'normales_caras.png',
    'normales_vertices.png',
    'flat_vs_smooth.png',
    'validacion_normales.png',
    'vedo_flat_vs_smooth.png',
    'vedo_normales_flechas_final.png',
]

with zipfile.ZipFile('outputs_normales.zip', 'w') as zf:
    for archivo in archivos:
        if os.path.exists(archivo):
            zf.write(archivo)
            print(f'   ✅ {archivo}')
        else:
            print(f'   ⚠️  {archivo} no encontrado — omitido')

from google.colab import files
files.download('outputs_normales.zip')
print('📦 ZIP descargado.')